In [2]:
!pip install tatoebatools
from tatoebatools import tatoeba

In [3]:
"""
Create a dictionary of each sentence and translation, categorised by the ID
"""
def get_sentence_dict(language:str):
  database = {}
  sentences = [s for s in tatoeba.sentences_detailed(language)]
  for sentence in sentences:
    database[sentence.sentence_id] = sentence.text

  return database

spanish_database = get_sentence_dict("spa")
english_database = get_sentence_dict("eng")

100%|██████████| 8.80M/8.80M [00:01<00:00, 7.11MiB/s]
100%|██████████| 33.5M/33.5M [00:02<00:00, 15.5MiB/s]


In [4]:
def get_sentences_with_translation(source_db:dict, target_db:dict, source_lang:str, target_lang:str):
  sentence_list = []
  sentences = [(lk.sentence_id, lk.translation_id) for lk in tatoeba.links(source_lang, target_lang)]

  for sentence in sentences:
    try:
      source_sentence = source_db[sentence[0]]
      target_sentence = target_db[sentence[1]]
      sentence_list.append([source_sentence, target_sentence])
    except KeyError:
      pass
  return sentence_list

sentence_list = get_sentences_with_translation(english_database, spanish_database, "eng", "spa")

100%|██████████| 1.68M/1.68M [00:00<00:00, 2.09MiB/s]


In [9]:
def get_sentences_with_word_in(word:str, index_to_search:int):
  for sentence in sentence_list:
    if f" {word} " in sentence[index_to_search]:
      return sentence[0] + ". " + sentence[1]

  return "No sentence found!"


print (get_sentences_with_word_in("hablar", 1))

May I talk to Ms. Brown?. ¿Puedo hablar con la Sra. Brown?


In [10]:
import requests

def get_top_2000_spanish_words():
    """
    Fetches the top 2000 most common Spanish words from Mark Davies' frequency list.
    Returns a list of words sorted by frequency in descending order.
    """
    url = "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/es/es_50k.txt"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception("Failed to download the frequency list.")

    lines = response.text.strip().split('\n')
    top_2000 = [line.split()[0] for line in lines[:2000]]
    return top_2000


In [13]:
top_words = get_top_2000_spanish_words()
string_to_save = ""

for word in top_words:
  sentence = get_sentences_with_word_in(word, 1)
  if sentence != "No sentence found!":
    sentence = sentence.replace(f" {word} ", " ____ ")
    sentence = sentence.replace(",", "")
    string_to_save += f"{word},{sentence}\n"

with open("topSpanishWordsAndSentences.csv", "w") as file:
  file.write(string_to_save)